Poutine

In [1]:
import os
import json
import subprocess
from pathlib import Path
from collections import defaultdict


# --------------------------------------------------
# Project paths
# --------------------------------------------------

ROOT_DIR = Path("/home/huawei/BOOSTSECURITY/42")

WORKFLOW_DIR = (
    ROOT_DIR
    / ".github"
    / "workflows"
)

POUTINE = (
    ROOT_DIR
    / "scanners"
    / "poutine"
    / "poutine"
)

RAW_JSON = (
    ROOT_DIR
    / "scanners_output"
    / "poutine"
    / "findings.json"
)

OUTPUT_DIR = (
    ROOT_DIR
    / "scanners_output"
    / "poutine"
    / "workflow_with_issues"
)


# --------------------------------------------------
# Check paths
# --------------------------------------------------

print("Current notebook directory :", Path.cwd())
print("ROOT_DIR                  :", ROOT_DIR)
print("WORKFLOW_DIR              :", WORKFLOW_DIR)
print("POUTINE                   :", POUTINE)
print("RAW_JSON                  :", RAW_JSON)
print("OUTPUT_DIR                :", OUTPUT_DIR)

print()
print("Root exists      :", ROOT_DIR.exists())
print("Workflows exist :", WORKFLOW_DIR.exists())
print("Poutine exists   :", POUTINE.exists())


if not ROOT_DIR.exists():
    raise FileNotFoundError(
        f"Project root not found: {ROOT_DIR}"
    )

if not WORKFLOW_DIR.exists():
    raise FileNotFoundError(
        f"Workflow directory not found: {WORKFLOW_DIR}"
    )

if not POUTINE.exists():
    raise FileNotFoundError(
        f"Poutine executable not found: {POUTINE}"
    )
# --------------------------------------------------
# 1. Run Poutine once and save raw JSON
# --------------------------------------------------

RAW_JSON.parent.mkdir(
    parents=True,
    exist_ok=True
)

try:
    completed = subprocess.run(
        [
            str(POUTINE),
            "analyze_local",
            str(ROOT_DIR),
            "--format",
            "json"
        ],
        capture_output=True,
        text=True,
        cwd=ROOT_DIR
    )

    stdout = (
        completed.stdout or ""
    ).strip()

    stderr = (
        completed.stderr or ""
    ).strip()


    if stderr:
        print("\nPoutine stderr:")
        print(stderr)


    if not stdout:
        print(
            "Poutine produced no output, "
            "nothing to parse."
        )
        raise SystemExit(0)


    with open(
        RAW_JSON,
        "w",
        encoding="utf-8"
    ) as f_out:
        f_out.write(stdout)


except Exception as e:
    raise SystemExit(
        f"Error running Poutine: {e}"
    )


# --------------------------------------------------
# 2. Create output directory
# --------------------------------------------------

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Optional but recommended:
# remove old per-workflow results from previous runs
for old_file in OUTPUT_DIR.glob("*.json"):
    old_file.unlink()


# --------------------------------------------------
# 3. Load raw findings
# --------------------------------------------------

try:
    with open(
        RAW_JSON,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

except Exception as e:
    raise SystemExit(
        f"Error reading/parsing {RAW_JSON}: {e}"
    )


all_findings = data.get(
    "findings",
    []
)


print(
    f"\nPoutine returned "
    f"{len(all_findings)} findings."
)


# --------------------------------------------------
# 4. Group findings by workflow filename
# --------------------------------------------------

grouped = defaultdict(list)


for finding in all_findings:

    meta = finding.get(
        "meta",
        {}
    )

    path = meta.get(
        "path"
    )


    if not path:
        continue


    workflow_file = Path(
        path
    ).name


    grouped[
        workflow_file
    ].append(
        {
            "rule_id":
                finding.get(
                    "rule_id",
                    ""
                ),

            "line":
                int(meta["line"])
                if (
                    "line" in meta
                    and str(meta["line"]).isdigit()
                )
                else meta.get(
                    "line",
                    ""
                ),

            "column":
                "",

            "job":
                meta.get(
                    "job",
                    ""
                ),

            "step":
                meta.get(
                    "step",
                    ""
                ),

            "details":
                meta.get(
                    "details",
                    ""
                ),

            "event_triggers":
                meta.get(
                    "event_triggers",
                    []
                ),
        }
    )


print(
    f"Findings belong to "
    f"{len(grouped)} workflow files."
)


# --------------------------------------------------
# 5. Write one JSON file per workflow with findings
# --------------------------------------------------

total_files_with_findings = 0


for file in sorted(
    os.listdir(WORKFLOW_DIR)
):

    if not file.endswith(
        (".yml", ".yaml")
    ):
        continue


    findings = grouped.get(
        file,
        []
    )


    if not findings:
        continue


    # Count findings by rule
    by_rule = {}


    for finding in findings:

        rule_id = finding.get(
            "rule_id",
            ""
        )

        if rule_id:

            by_rule[
                rule_id
            ] = (
                by_rule.get(
                    rule_id,
                    0
                )
                + 1
            )


    output_data = {
        "workflow": file,

        "tool": "poutine",

        "summary": {
            "total_findings":
                len(findings),

            "by_rule":
                by_rule
        },

        "findings":
            findings
    }


    output_path = (
        OUTPUT_DIR
        / f"{file}.json"
    )


    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f_out:

        json.dump(
            output_data,
            f_out,
            indent=2
        )


    total_files_with_findings += 1


# --------------------------------------------------
# 6. Summary
# --------------------------------------------------

print()
print("Poutine processing complete")
print("---------------------------")

print(
    f"Raw findings saved to: "
    f"{RAW_JSON}"
)

print(
    f"Total findings: "
    f"{len(all_findings)}"
)

print(
    f"Workflows with findings according to Poutine: "
    f"{len(grouped)}"
)

print(
    f"Per-workflow JSON files saved: "
    f"{total_files_with_findings}"
)

print(
    f"Output directory: "
    f"{OUTPUT_DIR}"
)

Current notebook directory : /home/huawei/BOOSTSECURITY/42/scripts
ROOT_DIR                  : /home/huawei/BOOSTSECURITY/42
WORKFLOW_DIR              : /home/huawei/BOOSTSECURITY/42/.github/workflows
POUTINE                   : /home/huawei/BOOSTSECURITY/42/scanners/poutine/poutine
RAW_JSON                  : /home/huawei/BOOSTSECURITY/42/scanners_output/poutine/findings.json
OUTPUT_DIR                : /home/huawei/BOOSTSECURITY/42/scanners_output/poutine/workflow_with_issues

Root exists      : True
Workflows exist : True
Poutine exists   : True

Poutine returned 233 findings.
Findings belong to 83 workflow files.

Poutine processing complete
---------------------------
Raw findings saved to: /home/huawei/BOOSTSECURITY/42/scanners_output/poutine/findings.json
Total findings: 233
Workflows with findings according to Poutine: 83
Per-workflow JSON files saved: 83
Output directory: /home/huawei/BOOSTSECURITY/42/scanners_output/poutine/workflow_with_issues


Zizmor


In [2]:
import os
import subprocess
from pathlib import Path


# --------------------------------------------------
# Paths
# --------------------------------------------------

ROOT_DIR = Path("/home/huawei/BOOSTSECURITY/42")

WORKFLOW_DIR = (
    ROOT_DIR
    / ".github"
    / "workflows"
)

ZIZMOR = (
    ROOT_DIR
    / "scanners"
    / "zizmor"
    / "zizmor"
)

RAW_DIR = (
    ROOT_DIR
    / "scanners_output"
    / "zizmor"
    / "raw_logs"
)


# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

print("ROOT_DIR     :", ROOT_DIR)
print("WORKFLOW_DIR :", WORKFLOW_DIR)
print("ZIZMOR       :", ZIZMOR)
print("RAW_DIR      :", RAW_DIR)

print()
print("Root exists      :", ROOT_DIR.exists())
print("Workflows exist  :", WORKFLOW_DIR.exists())
print("Zizmor exists    :", ZIZMOR.exists())


if not WORKFLOW_DIR.exists():
    raise FileNotFoundError(
        f"Workflow directory not found: {WORKFLOW_DIR}"
    )

if not ZIZMOR.exists():
    raise FileNotFoundError(
        f"Zizmor executable not found: {ZIZMOR}"
    )


RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# Run Zizmor on every workflow
# --------------------------------------------------

total = 0

for file in sorted(
    os.listdir(WORKFLOW_DIR)
):

    if not file.endswith(
        (".yml", ".yaml")
    ):
        continue


    file_path = (
        WORKFLOW_DIR
        / file
    )

    out_path = (
        RAW_DIR
        / f"{file}.txt"
    )


    try:

        with open(
            out_path,
            "w",
            encoding="utf-8"
        ) as out:

            subprocess.run(
                [
                    str(ZIZMOR),
                    str(file_path)
                ],
                stdout=out,

                # Keep stderr in the same log instead of
                # throwing it away
                stderr=subprocess.STDOUT,

                text=True,
                cwd=ROOT_DIR
            )


        total += 1

        print(
            f"[{total}] Saved log: {file}"
        )


    except Exception as e:

        print(
            f"Failed on {file}: {e}"
        )


print()
print(
    f"Finished running Zizmor on "
    f"{total} workflows."
)

print(
    f"Raw logs saved to: {RAW_DIR}"
)

ROOT_DIR     : /home/huawei/BOOSTSECURITY/42
WORKFLOW_DIR : /home/huawei/BOOSTSECURITY/42/.github/workflows
ZIZMOR       : /home/huawei/BOOSTSECURITY/42/scanners/zizmor/zizmor
RAW_DIR      : /home/huawei/BOOSTSECURITY/42/scanners_output/zizmor/raw_logs

Root exists      : True
Workflows exist  : True
Zizmor exists    : True
[1] Saved log: AOMediaCodec-libavif-ci-c-warnings.yml
[2] Saved log: AdobeDocs-analytics.es-ES-assign-issue.yml
[3] Saved log: AdobeDocs-audience-manager.ja-JP-assign-issue.yml
[4] Saved log: AdobeDocs-campaign.ko-KR-validate-articles.yml
[5] Saved log: AdobeDocs-customer-journey-analytics-learn.es-ES-assign-pr.yml
[6] Saved log: AdobeDocs-experience-manager-document-security.fr-FR-assign-issue.yml
[7] Saved log: AdobeDocs-experience-manager-guides-learn.ko-KR-assign-issue.yml
[8] Saved log: AdobeDocs-ffs-audio-video-API-lint.yml
[9] Saved log: AdobeDocs-marketo-developer.de-DE-assign-pr.yml
[10] Saved log: AdobeDocs-workfront.zh-Hans-assign-pr.yml
[11] Saved log: A

In [3]:
import re
import json
from pathlib import Path


# --------------------------------------------------
# Paths
# --------------------------------------------------

ROOT_DIR = Path("/home/huawei/BOOSTSECURITY/42")

input_dir = (
    ROOT_DIR
    / "scanners_output"
    / "zizmor"
    / "raw_logs"
)

output_dir = (
    ROOT_DIR
    / "scanners_output"
    / "zizmor"
    / "workflow_with_issues"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# Check paths
# --------------------------------------------------

print("Input directory :", input_dir)
print("Output directory:", output_dir)
print("Input exists    :", input_dir.exists())

raw_files = list(input_dir.glob("*.txt"))

print("Raw log files   :", len(raw_files))


# --------------------------------------------------
# Regex patterns
# --------------------------------------------------

ansi_escape = re.compile(
    r"\x1B\[[0-?]*[ -/]*[@-~]"
)

osc8_link = re.compile(
    r"\x1b]8;;[^\x1b]*\x1b\\(.*?)\x1b]8;;\x1b\\"
)

finding_pattern = re.compile(
    r"(info|warning|error)\[(.+?)\]: (.+)"
)

# Accept:
#
# .github/workflows/file.yml
# ../.github/workflows/file.yml
# /home/.../.github/workflows/file.yml
#
location_pattern = re.compile(
    r"-->\s+.*?\.github/workflows/(.+?):(\d+):(\d+)"
)

note_pattern = re.compile(
    r"= note: (.+)",
    re.IGNORECASE
)


# --------------------------------------------------
# Parse raw logs
# --------------------------------------------------

saved_files = 0


for input_file in input_dir.glob("*.txt"):

    findings = []
    current = {}

    workflow_file = input_file.stem


    with open(
        input_file,
        encoding="utf-8"
    ) as f:

        raw_lines = f.readlines()


    # --------------------------------------------------
    # Strip ANSI & hyperlinks
    # --------------------------------------------------

    clean_lines = []

    for line in raw_lines:

        line = ansi_escape.sub(
            "",
            line
        )

        line = osc8_link.sub(
            r"\1",
            line
        )

        clean_lines.append(
            line.strip()
        )


    # --------------------------------------------------
    # Skip workflows with no findings
    # --------------------------------------------------

    if any(
        "No findings to report" in line
        for line in clean_lines
    ):
        continue


    # --------------------------------------------------
    # Parse findings
    # --------------------------------------------------

    for line in clean_lines:

        if not line:
            continue


        m = finding_pattern.match(
            line
        )

        if m:

            if current.get(
                "rule_id"
            ):
                findings.append(
                    current
                )


            level, rule_id, message = (
                m.groups()
            )


            rule_id = osc8_link.sub(
                r"\1",
                rule_id
            )


            current = {
                "rule_id":
                    rule_id.strip(),

                "level":
                    level,

                "message":
                    message.strip(),

                "workflow":
                    workflow_file,

                "line":
                    None,

                "column":
                    None,

                "note":
                    None
            }

            continue


        m = location_pattern.match(
            line
        )

        if m:

            _, line_num, col_num = (
                m.groups()
            )

            current[
                "line"
            ] = int(
                line_num
            )

            current[
                "column"
            ] = int(
                col_num
            )

            continue


        m = note_pattern.match(
            line
        )

        if m:

            current[
                "note"
            ] = (
                m.group(1)
                .strip()
            )

            continue


    # --------------------------------------------------
    # Add last finding
    # --------------------------------------------------

    if current.get(
        "rule_id"
    ):
        findings.append(
            current
        )


    if not findings:
        continue


    # --------------------------------------------------
    # Summary by rule
    # --------------------------------------------------

    rule_summary = {}

    for f in findings:

        rule_summary[
            f["rule_id"]
        ] = (
            rule_summary.get(
                f["rule_id"],
                0
            )
            + 1
        )


    # --------------------------------------------------
    # Result
    # --------------------------------------------------

    result = {
        "workflow":
            workflow_file,

        "tool":
            "zizmor",

        "summary": {
            "total_findings":
                len(findings),

            "by_rule":
                rule_summary
        },

        "findings":
            findings
    }


    # --------------------------------------------------
    # Save result
    # --------------------------------------------------

    out_path = (
        output_dir
        / f"{workflow_file}.json"
    )


    with open(
        out_path,
        "w",
        encoding="utf-8"
    ) as out:

        json.dump(
            result,
            out,
            indent=2
        )


    saved_files += 1

    print(
        f"{workflow_file} -> "
        f"{len(findings)} findings"
    )


# --------------------------------------------------
# Final summary
# --------------------------------------------------

print()
print(
    f"Parsed {saved_files} workflow "
    f"result files to {output_dir}"
)

Input directory : /home/huawei/BOOSTSECURITY/42/scanners_output/zizmor/raw_logs
Output directory: /home/huawei/BOOSTSECURITY/42/scanners_output/zizmor/workflow_with_issues
Input exists    : True
Raw log files   : 475
fortinet-fortisoar-connector-unit-42-intel-objects-feed-pytest_checks.yml -> 2 findings
Sundsvallskommun-api-service-garbage-dependabot-reviewer.yml -> 2 findings
discourse-discourse-highlightjs-structured-text-discourse-theme.yml -> 1 findings
symfony-zulip-notifier-close-pull-request.yml -> 3 findings
simp-pupmod-simp-swap-pr_tests.yml -> 19 findings
Geonovum-3d-tiling-build.yml -> 5 findings
ministryofjustice-hmpps-nomis-mapping-service-pipeline.yml -> 7 findings
zopencommunity-pvport-build_and_test.yml -> 2 findings
dpc-sdp-tide_demo_content-build.yml -> 9 findings
Geonovum-HR-CRS-Gebruik-main.yml -> 3 findings
canonical-smartctl-exporter-snap-promote.yaml -> 6 findings
npm-hosted-git-info-release.yml -> 18 findings
AdobeDocs-campaign.ko-KR-validate-articles.yml -> 1 f

Normalize the findings 

In [4]:
import json
from pathlib import Path
from collections import defaultdict


# --------------------------------------------------
# Paths
# --------------------------------------------------

ROOT_DIR = Path("/home/huawei/BOOSTSECURITY/42")

SCANNERS_OUTPUT_DIR = (
    ROOT_DIR
    / "scanners_output"
)

TOOLS = [
    "poutine",
    "zizmor"
]

base_dir = SCANNERS_OUTPUT_DIR

output_dir = (
    SCANNERS_OUTPUT_DIR
    / "merged"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# Merge results
# --------------------------------------------------

workflow_results = defaultdict(dict)


for tool in TOOLS:

    tool_dir = (
        base_dir
        / tool
        / "workflow_with_issues"
    )

    if not tool_dir.exists():
        print(
            f"[!] Directory not found: {tool_dir}"
        )
        continue


    for wf_file in tool_dir.glob("*.json"):

        # Normalize to unify .yml/.yaml
        workflow_name = wf_file.stem

        if workflow_name.endswith(".yaml"):
            workflow_name = workflow_name.replace(
                ".yaml",
                ".yml"
            )


        with open(
            wf_file,
            "r",
            encoding="utf-8"
        ) as f:

            tool_result = json.load(f)

            workflow_results[
                workflow_name
            ][tool] = tool_result


# --------------------------------------------------
# Save merged results
# --------------------------------------------------

for wf_name, tool_dict in workflow_results.items():

    merged = {
        "workflow": wf_name,
        "tools": tool_dict
    }


    with open(
        output_dir / f"{wf_name}.json",
        "w",
        encoding="utf-8"
    ) as out_f:

        json.dump(
            merged,
            out_f,
            indent=2
        )


print(
    f"[✓] Saved {len(workflow_results)} "
    f"merged workflow files to {output_dir}"
)

[✓] Saved 454 merged workflow files to /home/huawei/BOOSTSECURITY/42/scanners_output/merged


In [5]:
import json
import csv
from pathlib import Path

# --------------------------------------------------
# Paths
# --------------------------------------------------

ROOT_DIR = Path("/home/huawei/BOOSTSECURITY/42")

INPUT_DIR = ROOT_DIR / "scanners_output" / "merged"
OUTPUT_DIR = ROOT_DIR / "scanners_output" / "normalized"
MAPPING_FILE = ROOT_DIR / "scanners" / "rules_mapping.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOOLS = {"poutine", "zizmor"}


# --------------------------------------------------
# Load rule -> weakness mapping
# --------------------------------------------------

rule_map = {}

with open(MAPPING_FILE, newline="", encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        tool = row["tool_name"].strip().lower()

        if tool not in TOOLS:
            continue

        rule = row["rule"].strip().lower()
        weakness = row["weakness"].strip()

        rule_map[(tool, rule)] = weakness


# --------------------------------------------------
# Normalize merged files
# --------------------------------------------------

normalized_files = 0
mapped_findings = 0
unmapped_findings = 0
unmapped_rules = set()


for file_path in INPUT_DIR.glob("*.json"):

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for tool, result in data.get("tools", {}).items():

        tool_name = tool.strip().lower()

        if tool_name not in TOOLS:
            continue

        findings = result.get("findings", [])

        for finding in findings:

            rule_id = (
                finding.get("rule_id")
                or finding.get("rule")
                or finding.get("name")
                or ""
            )

            rule_id = str(rule_id).strip().lower()

            capability = rule_map.get(
                (tool_name, rule_id)
            )

            if capability:
                finding["capability"] = capability
                mapped_findings += 1

            else:
                unmapped_findings += 1
                unmapped_rules.add(
                    (tool_name, rule_id)
                )

    # Save normalized file
    output_file = OUTPUT_DIR / file_path.name

    with open(output_file, "w", encoding="utf-8") as out_f:
        json.dump(
            data,
            out_f,
            indent=2
        )

    normalized_files += 1


# --------------------------------------------------
# Summary
# --------------------------------------------------

print(f"[✓] Normalized files : {normalized_files}")
print(f"[✓] Mapped findings  : {mapped_findings}")
print(f"[!] Unmapped findings: {unmapped_findings}")
print(f"[✓] Saved to         : {OUTPUT_DIR}")


if unmapped_rules:
    print("\nUnmapped rules:")

    for tool, rule in sorted(unmapped_rules):
        print(f"  {tool}: {rule}")

[✓] Normalized files : 454
[✓] Mapped findings  : 2237
[!] Unmapped findings: 0
[✓] Saved to         : /home/huawei/BOOSTSECURITY/42/scanners_output/normalized
